In [ ]:
import pandas as pd
from fastparquet import write
from fastparquet import ParquetFile
from sklearn.preprocessing import KBinsDiscretizer
from pathlib import Path
import os

In [ ]:
data_pipeline = "bins"
input_pipeline = "impute"
bin_strategy = "uniform"
file_extension_name = "baseline"
n_bins = 20

In [29]:
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    data_path = "../../kaggle/input/datasets/abhinavneelam/smartphone-addiction/data/"
    output_path = "/kaggle/working/"
else:
    data_path = "../../data/"
    output_path = "../../"

In [30]:
experiment_path = Path(data_path) / f"{data_pipeline}"
experiment_path.mkdir(parents=True, exist_ok=True)

In [31]:
ss = pd.read_csv("../../data/raw/sample_submission.csv")
target_column = ss.columns[-1]
target_column

'addicted_label'

In [32]:
X = ParquetFile(Path(data_path) / f"{input_pipeline}/train_{file_extension_name}.parq").to_pandas()
X_test = ParquetFile(Path(data_path) / f"{input_pipeline}/test_{file_extension_name}.parq").to_pandas()

X.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   age                      691369 non-null  float64
 1   daily_screen_time_hours  691369 non-null  float64
 2   social_media_hours       691369 non-null  float64
 3   gaming_hours             691369 non-null  float64
 4   work_study_hours         691369 non-null  float64
 5   sleep_hours              691369 non-null  float64
 6   notifications_per_day    691369 non-null  float64
 7   app_opens_per_day        691369 non-null  float64
 8   weekend_screen_time      691369 non-null  float64
 9   stress_level             691369 non-null  float64
 10  academic_work_impact     691369 non-null  bool   
 11  Female                   691369 non-null  bool   
 12  Male                     691369 non-null  bool   
 13  Other                    691369 non-null  bool   
dtypes: bool(4), flo

In [33]:
drop_columns = X.columns
cat_columns = X.select_dtypes(include=['category']).columns.to_list()
num_columns = X.select_dtypes(include=['float64']).columns.to_list()

est = KBinsDiscretizer(n_bins=n_bins, strategy=bin_strategy, encode='ordinal')
X_bins = est.fit_transform(X[num_columns])
X_test_bins = est.transform(X_test[num_columns])

X = pd.DataFrame(X_bins, columns=num_columns)
X = X.add_suffix(f"_{bin_strategy}_bins_{n_bins}")
X_test = pd.DataFrame(X_test_bins, columns=num_columns)
X_test = X_test.add_suffix(f"_{bin_strategy}_bins_{n_bins}")

X.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 10 columns):
 #   Column                                   Non-Null Count   Dtype  
---  ------                                   --------------   -----  
 0   age_uniform_bins_20                      691369 non-null  float64
 1   daily_screen_time_hours_uniform_bins_20  691369 non-null  float64
 2   social_media_hours_uniform_bins_20       691369 non-null  float64
 3   gaming_hours_uniform_bins_20             691369 non-null  float64
 4   work_study_hours_uniform_bins_20         691369 non-null  float64
 5   sleep_hours_uniform_bins_20              691369 non-null  float64
 6   notifications_per_day_uniform_bins_20    691369 non-null  float64
 7   app_opens_per_day_uniform_bins_20        691369 non-null  float64
 8   weekend_screen_time_uniform_bins_20      691369 non-null  float64
 9   stress_level_uniform_bins_20             691369 non-null  float64
dtypes: float64(10)
memory usage: 52.7 MB


In [34]:
write(experiment_path / f"train_{bin_strategy}_bins_{n_bins}.parq", X)
write(experiment_path / f"test_{bin_strategy}_bins_{n_bins}.parq", X_test)